In [1]:
# Logistic Regression(RGB128×128,PCA 64)

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
from sklearn.linear_model import LogisticRegression

In [32]:
seed = 90051
image_size = (128, 128)
pca_dimention = 64
fixed_batch_size = 128
hype_candidates = [0.1, 1.0, 10.0]
resumed_midway = True
max_iteration = 500
test_condistion_list = ['clean', 'upper_mask', 'lower_mask', 'blur', 'noise']
mask_bright_value = 128
blurr_std = 1.0
blurr_radius = 3
noise_std = 0.05

In [33]:
CSV_FILE_PATH = Path('expw_subset_metadata_comp90051.csv')
IMAGE_ROOT_PATH = Path('ExpW_extracted\origin')
OUTER_FILE_PATH = Path('Outer_10-fold_CV_splits.json')
INNER_FILE_PATH = Path('Inner_3-fold_CV_splits.json')
RGB_CACHE_FILE_PATH = Path('expw_rgb128_uint8.npy')
RESULTS_FILE_PATH = Path('lr_robustness_results_rgb128.csv')
SUMMARY_FILE_PATH = Path('lr_robustness_summary_rgb128.csv')

In [34]:
# Load metadata and shared CV splits

In [35]:
#Check file exsistence
if not CSV_FILE_PATH.is_file():
    raise FileNotFoundError(CSV_FILE_PATH)
if not OUTER_FILE_PATH.is_file():
    raise FileNotFoundError(OUTER_FILE_PATH)
if not INNER_FILE_PATH.is_file():
    raise FileNotFoundError(INNER_FILE_PATH)
if not IMAGE_ROOT_PATH.is_dir():
    raise FileNotFoundError(IMAGE_ROOT_PATH)

In [36]:
# Load metadata and inner and outer CV splits
df = pd.read_csv(CSV_FILE_PATH)
y = df['correct_facial_class'].to_numpy(dtype=int)
with open(OUTER_FILE_PATH, 'r', encoding='utf-8') as f:
    outer_splits = json.load(f)
with open(INNER_FILE_PATH, 'r', encoding='utf-8') as f:
    inner_splits = json.load(f)

In [37]:
# Checks Data contents
if len(df) != 14935:
    raise AssertionError("The number of data does not match the number planned for this analysis.")
if len(outer_splits) != 10:
    raise AssertionError("The number of outer folds for this research does not match 10.")
if len(inner_splits) != 30:
    raise AssertionError("The number of inner folds for this search does not match 30.")
print("Number of samples:", len(df),"\nOuter folds:", len(outer_splits),"\nInner folds:", len(inner_splits))

Number of samples: 14935 
Outer folds: 10 
Inner folds: 30


In [38]:
# RGB128 crop and perturbations

In [39]:
def face_image_preprocessing_function(row_index):
    row = df.iloc[int(row_index)]
    image_name = str(row['original_image_name'])
    image_path = IMAGE_ROOT_PATH / image_name
    with Image.open(image_path) as img:
        img = img.convert('RGB')
        left = int(row['left'])
        top = int(row['top'])
        right = int(row['right'])
        bottom = int(row['bottom'])
        if left < 0 or top < 0:
            raise ValueError("Invalid bounding box")
        if right > img.width or bottom > img.height:
            raise ValueError("Invalid bounding box")
        if left >= right or top >= bottom:
            raise ValueError("Invalid bounding box")
        width = right - left
        height = bottom - top
        if width < 100 or height < 100:
            raise ValueError("Face crop is smaller than 100x100")
        face = img.crop((left, top, right, bottom))
        face = face.resize(image_size,Image.Resampling.BILINEAR)
        face_array = np.array(face, dtype=np.uint8)
    return face_array

In [40]:
def build_or_load_rgb_cache():
    expected_shape = (len(df), image_size[1], image_size[0],3)
    if RGB_CACHE_FILE_PATH.is_file():
        cache = np.load(RGB_CACHE_FILE_PATH,mmap_mode='r')
        if cache.shape != expected_shape:
            raise ValueError("The existing cache has the wrong shape: "+ str(cache.shape))
        if cache.dtype != np.uint8:
            raise ValueError("The existing cache has the wrong data type: "+ str(cache.dtype))
        print("Using existing RGB128 cache")
        return cache
    print("Building RGB128 cache...")
    print("This may take a while the first time.")
    cache = np.lib.format.open_memmap(RGB_CACHE_FILE_PATH, mode='w+', dtype=np.uint8,shape=expected_shape)
    for i in range(len(df)):
        image = face_image_preprocessing_function(i)
        cache[i] = image
        cache.flush()
    cache.flush()
    del cache
    print("RGB128 cache completed")
    cache = np.load(RGB_CACHE_FILE_PATH,mmap_mode='r')
    return cache

In [41]:
RGB_CACHE = build_or_load_rgb_cache()
print("Cache shape:", RGB_CACHE.shape, "\nCache type:",RGB_CACHE.dtype)

Using existing RGB128 cache
Cache shape: (14935, 128, 128, 3) 
Cache type: uint8


In [42]:
# Batch loading and fold-specific PCA

In [43]:
def batches(indices, minimum=1):
    indices = np.asarray(indices, dtype=int)
    chunks = []
    start = 0
    while start < len(indices):
        end = start + fixed_batch_size
        batch = indices[start:end]
        chunks.append(batch)
        start = end
    if len(chunks) > 1:
        last_batch = chunks[-1]
        if len(last_batch) < minimum:
            previous_batch = chunks[-2]
            combined_batch = np.concatenate((previous_batch, last_batch))
            chunks[-2] = combined_batch
            chunks.pop()
    return chunks

In [44]:
def load_batch(indices, condition='clean'):
    indices = np.asarray(indices, dtype=int)
    images = RGB_CACHE[indices]
    images = np.asarray(images, dtype=np.float32)
    images = images / 255.0
    if condition == 'upper_mask':
        middle = images.shape[1] // 2
        grey_value = mask_bright_value / 255.0
        images[:, :middle, :, :] = grey_value
    elif condition == 'lower_mask':
        middle = images.shape[1] // 2
        grey_value = mask_bright_value / 255.0
        images[:, middle:, :, :] = grey_value
    elif condition == 'blur':
        for i in range(len(images)):
            blurred_image = gaussian_filter(images[i], sigma=blurr_std,radius=blurr_radius,axes=(0, 1))
            images[i] = blurred_image
    elif condition == 'noise':
        for i in range(len(indices)):
            row_index = int(indices[i])
            random_seed = np.random.SeedSequence([seed, row_index])
            rng = np.random.default_rng(random_seed)
            noise = rng.normal(0.0,noise_std, images[i].shape)
            noise = noise.astype(np.float32)
            noisy_image = images[i] + noise
            images[i] = np.clip(noisy_image,0.0, 1.0 )
    elif condition != 'clean':
        raise ValueError( "Unknown condition: " + str(condition))
    images = images.reshape(len(indices),-1)
    return images

In [45]:
def fit_preprocessor(train_indices, label='preprocessor'):
    train_indices = np.asarray(train_indices, dtype=int)
    rng = np.random.default_rng(seed)
    order = rng.permutation(train_indices)
    scaler_batches = batches(order)
    pca_batches = batches(order, minimum=pca_dimention)
    print(label,"- fitting scaler with",len(order),"samples")
    scaler = StandardScaler()
    start_time = time.perf_counter()
    for batch_no, batch in enumerate(scaler_batches, start=1):
        x_batch = load_batch(batch, 'clean')
        scaler.partial_fit(x_batch)
        if batch_no % 20 == 0 or batch_no == len(scaler_batches):
            print("Scaler:",batch_no,"/", len(scaler_batches),"batches")
    scaler_time = (time.perf_counter() - start_time) / 60
    print("Scaler finished in", round(scaler_time, 1),"minutes")
    print(label,"- fitting PCA")
    pca = IncrementalPCA(n_components=pca_dimention,batch_size=fixed_batch_size)
    start_time = time.perf_counter()
    for batch_no, batch in enumerate(pca_batches, start=1):
        x_batch = load_batch(batch, 'clean')
        x_batch = scaler.transform(x_batch)
        pca.partial_fit(x_batch)
        if batch_no % 20 == 0 or batch_no == len(pca_batches):
            elapsed = (time.perf_counter() - start_time) / 60
            print("PCA:", batch_no, "/", len(pca_batches), "batches -",round(elapsed, 1), "minutes")
    pca_time = ( time.perf_counter() - start_time) / 60
    print("PCA finished in",round(pca_time, 1),"minutes")
    return scaler, pca

In [46]:
def transform_indices(indices,scaler,pca,condition='clean',label='transform'):
    indices = np.asarray(indices,dtype=int)
    chunk_list = batches(indices)
    scores = np.empty((len(indices), pca_dimention),dtype=np.float32)
    position = 0
    print(label,"- transforming", len(indices),"samples for",condition)
    start_time = time.perf_counter()
    for batch_no, batch in enumerate(chunk_list, start=1):
        values = load_batch(batch,condition)
        values = scaler.transform(values)
        transformed_values = pca.transform(values)
        end_position = position + len(batch)
        scores[position:end_position] = transformed_values
        position = end_position
        if batch_no % 20 == 0 or batch_no == len(chunk_list):
            print("Transform:", batch_no, "/", len(chunk_list), "batches")
    elapsed = (time.perf_counter() - start_time) / 60
    print("Transform finished in",round(elapsed, 1),"minutes")
    return scores

In [47]:
# Metrics computed from scratch

In [48]:
def manual_mean(values):
    values = list(values)
    if len(values) == 0:
        return 0.0
    total = 0.0
    for value in values:
        total += float(value)
    mean = total / len(values)
    return mean

In [49]:
def manual_std(values):
    values = list(values)
    if len(values) <= 1:
        return 0.0
    mean = manual_mean(values)
    squared_difference_sum = 0.0
    for value in values:
        difference = float(value) - mean
        squared_difference_sum += difference ** 2
    variance = squared_difference_sum / (len(values) - 1)
    standard_deviation = variance ** 0.5
    return standard_deviation

In [50]:
def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    correct = 0
    for i in range(len(y_true)):
        actual = int(y_true[i])
        predicted = int(y_pred[i])
        if actual == predicted:
            correct += 1
    accuracy = correct / len(y_true)
    f1_scores = []
    recall_scores = []
    for class_number in range(7):
        true_positive = 0
        false_positive = 0
        false_negative = 0
        for i in range(len(y_true)):
            actual = int(y_true[i])
            predicted = int(y_pred[i])
            if actual == class_number and predicted == class_number:
                true_positive += 1
            elif actual != class_number and predicted == class_number:
                false_positive += 1
            elif actual == class_number and predicted != class_number:
                false_negative += 1
        f1_denominator = (2 * true_positive + false_positive + false_negative)
        if f1_denominator == 0:
            f1 = 0.0
        else:
            f1 = (2 * true_positive/ f1_denominator)
        recall_denominator = (true_positive + false_negative)
        if recall_denominator == 0:
            recall = 0.0
        else:
            recall = (true_positive / recall_denominator)
        f1_scores.append(f1)
        recall_scores.append(recall)
    macro_f1 = manual_mean(f1_scores)
    macro_recall = manual_mean(recall_scores)
    results = {'accuracy': float(accuracy),'macro_f1': float(macro_f1),'macro_recall': float(macro_recall)}
    return results

In [51]:
def confusion_matrix_manual(y_true, y_pred):
    matrix = np.zeros((7, 7), dtype=int)
    for i in range(len(y_true)):
        actual = int(y_true[i])
        predicted = int(y_pred[i])
        matrix[actual, predicted] += 1
    return matrix

In [52]:
# Inner 3-fold hyputerparameter tuning for C

In [53]:
def tune_C(labels, outer_fold_id):
    selected = []
    for split in inner_splits:
        if int(split['outer_fold']) == int(outer_fold_id):
            selected.append(split)
    assert len(selected) == 3
    scores = {}
    for c in hype_candidates:
        scores[c] = []
    for inner_number, inner in enumerate(selected, start=1):
        inner_start = time.perf_counter()
        train_idx = np.asarray(inner['train_indices'],dtype=int)
        val_idx = np.asarray(inner['validation_indices'],dtype=int)
        print("\nInner fold",inner_number, "/ 3")
        print("Training samples:",len(train_idx),"| Validation samples:", len(val_idx))
        scaler, pca = fit_preprocessor(train_idx, label="outer " + str(outer_fold_id)+ " / inner " + str(inner_number))
        X_train = transform_indices(train_idx, scaler, pca,'clean', label="inner " + str(inner_number) + " train")
        X_val = transform_indices(val_idx, scaler, pca,'clean', label="inner " + str(inner_number) + " validation")
        pca_scaler = StandardScaler()
        X_train = pca_scaler.fit_transform(X_train)
        X_val = pca_scaler.transform(X_val)
        model = LogisticRegression(C=hype_candidates[0], solver='lbfgs',  max_iter=max_iteration,random_state=seed, warm_start=True)
        for c in hype_candidates:
            print("Testing C =", c)
            model.set_params(C=c)
            model.fit(X_train, labels[train_idx])
            predictions = model.predict(X_val)
            results = evaluate(labels[val_idx],predictions)
            macro_f1 = results['macro_f1']
            scores[c].append(macro_f1)
            print("Macro-F1:",round(macro_f1, 6))
        elapsed = (time.perf_counter() - inner_start) / 60
        print("Inner fold",inner_number, "finished in",round(elapsed, 1),"minutes")
    mean_scores = {}
    for c in hype_candidates:
        mean_scores[c] = manual_mean(scores[c])
    best_c = hype_candidates[0]
    best_score = mean_scores[best_c]
    for c in hype_candidates:
        if mean_scores[c] > best_score:
            best_c = c
            best_score = mean_scores[c]
    print("Average Macro-F1 scores:", mean_scores)
    print("Selected C:", best_c)
    return best_c, mean_scores

In [54]:
# Outer 10-fold evaluation

In [55]:
def save_results_and_summary(results):
    results_df = pd.DataFrame(results)
    results_df.to_csv(RESULTS_FILE_PATH, index=False)
    summary_rows = []
    for condition in test_condistion_list:
        condition_rows = []
        for row in results:
            if row['condition'] == condition:
                condition_rows.append(row)
        if len(condition_rows) == 0:
            continue
        accuracy_values = []
        f1_values = []
        recall_values = []
        for row in condition_rows:
            accuracy_values.append(row['accuracy'])
            f1_values.append(row['macro_f1'])
            recall_values.append(row['macro_recall'])
        accuracy_mean = manual_mean(accuracy_values)
        accuracy_std = manual_std(accuracy_values)
        f1_mean = manual_mean(f1_values)
        f1_std = manual_std(f1_values)
        recall_mean = manual_mean(recall_values)
        recall_std = manual_std(recall_values)
        summary = {'condition': condition, 'accuracy_mean': accuracy_mean,'accuracy_std': accuracy_std,
            'macro_f1_mean': f1_mean, 'macro_f1_std': f1_std, 'macro_recall_mean': recall_mean, 'macro_recall_std': recall_std}
        summary_rows.append(summary)
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(SUMMARY_FILE_PATH, index=False)
    return results_df, summary_df

In [56]:
results = []
if resumed_midway and RESULTS_FILE_PATH.is_file():
    previous_results = pd.read_csv(RESULTS_FILE_PATH)
    required_columns = {'fold','condition','best_C', 'accuracy', 'macro_f1', 'macro_recall'}
    if required_columns.issubset(previous_results.columns):
        results = previous_results.to_dict('records')
        print("Loaded", len(results), "saved result rows")

Loaded 50 saved result rows


In [57]:
completed_folds = set()
for fold_id in range(1, 11):
    rows_for_fold = []
    for row in results:
        if int(row['fold']) == fold_id:
            rows_for_fold.append(row)
    conditions_found = set()
    for row in rows_for_fold:
        conditions_found.add(row['condition'])
    all_conditions_done = set(test_condistion_list).issubset(conditions_found)
    if all_conditions_done:
        completed_folds.add(fold_id)

In [58]:
completed_times = []
for outer in outer_splits:
    fold_id = int(outer['fold'])
    if fold_id in completed_folds:
        print("\nOuter fold",fold_id,"already completed")
        continue
    new_results = []
    for row in results:
        if int(row['fold']) != fold_id:
            new_results.append(row)
    results = new_results
    fold_start = time.perf_counter()
    train_idx = np.asarray(outer['train_indices'],dtype=int)
    test_idx = np.asarray(outer['test_indices'],dtype=int)
    print("\n------------------------------")
    print("Outer fold", fold_id, "of 10")
    print("Training samples:", len(train_idx))
    print("Test samples:", len(test_idx))
    print("------------------------------")
    print("\nTuning C...")
    best_c, inner_scores = tune_C(y, fold_id)
    print( "Selected C:",best_c)
    print("Inner Macro-F1:",inner_scores)
    print("\nFitting preprocessing for outer fold",fold_id)
    scaler, pca = fit_preprocessor( train_idx, label="outer " + str(fold_id))
    X_train = transform_indices(train_idx,scaler,pca,'clean', label="outer " + str(fold_id) + " train")
    pca_scaler = StandardScaler()
    X_train = pca_scaler.fit_transform( X_train)
    print("\nTraining Logistic Regression with C =",best_c)
    model = LogisticRegression(C=best_c, solver='lbfgs', max_iter=max_iteration, random_state=seed)
    model.fit(X_train, y[train_idx])
    print("\nTesting image conditions...")
    for condition in test_condistion_list:
        print( "Condition:",condition)
        X_test = transform_indices(test_idx, scaler, pca, condition, label="outer " + str(fold_id) + " test")
        X_test = pca_scaler.transform(X_test)
        predictions = model.predict( X_test)
        metrics = evaluate( y[test_idx],predictions)
        result_row = {'fold': fold_id, 'condition': condition, 'best_C': best_c, 'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1'], 'macro_recall': metrics['macro_recall']}
        results.append(result_row)
        print("Accuracy:", round(metrics['accuracy'], 4),
            "| Macro-F1:", round(metrics['macro_f1'], 4),
            "| Macro-Recall:",round(metrics['macro_recall'], 4))
    results_df, summary_df = save_results_and_summary(results)
    fold_minutes = (time.perf_counter() - fold_start) / 60
    completed_times.append(fold_minutes)
    print( "\nOuter fold", fold_id, "finished in", round(fold_minutes, 1),"minutes")
    print("Saved",len(results_df),"result rows")


Outer fold 1 already completed

Outer fold 2 already completed

Outer fold 3 already completed

Outer fold 4 already completed

Outer fold 5 already completed

Outer fold 6 already completed

Outer fold 7 already completed

Outer fold 8 already completed

Outer fold 9 already completed

Outer fold 10 already completed


In [59]:
results_df, summary_df = save_results_and_summary(results)
print("Final results:",results_df.tail(10),
      "\nSummary:",summary_df,
      "\nExpected number of result rows:",10 * len(test_condistion_list))

Final results:     fold   condition  best_C  accuracy  macro_f1  macro_recall
40     9       clean     1.0  0.302076  0.232390      0.253641
41     9  upper_mask     1.0  0.258540  0.181072      0.212935
42     9  lower_mask     1.0  0.217013  0.140368      0.187378
43     9        blur     1.0  0.302076  0.232522      0.253641
44     9       noise     1.0  0.300737  0.230752      0.252331
45    10       clean     1.0  0.275285  0.210720      0.230803
46    10  upper_mask     1.0  0.236437  0.162485      0.194500
47    10  lower_mask     1.0  0.208305  0.140467      0.180213
48    10        blur     1.0  0.275954  0.211244      0.231353
49    10       noise     1.0  0.274615  0.210157      0.230254 
Summary:     condition  accuracy_mean  accuracy_std  macro_f1_mean  macro_f1_std  \
0       clean       0.287781      0.007961       0.221845      0.006381   
1  upper_mask       0.247540      0.009754       0.172312      0.006770   
2  lower_mask       0.217542      0.009030       0.147393